# Step 5: From Zero-Shot Baseline to Fine-Tuning

This notebook follows a structured workflow:
1. **Zero-Shot Evaluation**: Replicating Step 4 performance by mapping COCO predictions to Cityscapes Train IDs.
2. **Fine-Tuning Setup**: Debugging the resolution and class-head mismatches to prepare the model for native Cityscapes training.

## 1. Environment Setup

In [ ]:
!pip install lightning > /dev/null
!pip install gitignore_parser > /dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null

In [ ]:
from google.colab import drive
import os
import sys
import json
import yaml
import torch
import importlib
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightning import seed_everything

# 1. Mount Drive and Configure Paths
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

project_root = '/content/drive/MyDrive/FundGitHubProject'
if not os.path.exists('/content/ProjectFolder'):
  # creates shortcut to access the project folder
  !ln -s /content/drive/MyDrive/FundGitHubProject /content/ProjectFolder
eomt_folder = project_root + '/eomt'

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if eomt_folder not in sys.path:
    sys.path.insert(0, eomt_folder)

from eval.iouEval import iouEval
seed_everything(0, verbose=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

Mounted at /content/drive
Active Device: cuda


## 2. Zero-Shot Baseline (The Step 4 Score)
Before fine-tuning, we verify that our model loading and mapping logic achieve the same results as Step 4.

In [ ]:
from training.mask_classification_panoptic import MaskClassificationPanoptic
from models.eomt import EoMT
from models.vit import ViT

# 1. Load COCO Config and weights
coco_cfg_path = os.path.join(eomt_folder, 'configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml')
with open(coco_cfg_path, "r") as f: coco_config = yaml.safe_load(f)

coco_img_size = (640, 640)
bin_path = os.path.join(eomt_folder, 'eomt_weights/eomt_coco.bin')

# 2. Initialize Model (Standard COCO Configuration)
encoder = ViT(img_size=640, backbone_name="vit_base_patch14_reg4_dinov2")
network = EoMT(
    num_classes=133,
    encoder=encoder,
    num_q=200,
    num_blocks=3,
    masked_attn_enabled=False
)
model_coco = MaskClassificationPanoptic(
    network=network,
    img_size=(640, 640), # Changed from 640 to (640, 640)
    num_classes=133,
    stuff_classes=coco_config["data"].get("init_args", {}).get("stuff_classes", []),
    attn_mask_annealing_enabled=False
)

# Load weights
ckpt = torch.load(bin_path, map_location="cpu")
model_coco.load_state_dict(ckpt.get("state_dict", ckpt), strict=False)
model_coco.to(device).eval()
print("COCO Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


COCO Model loaded successfully!


In [ ]:
# 3. Setup Mapping (From Step 4)
map_file = os.path.join(project_root, 'coco-classes-mapping-master/coco_mapping_80to91.json')
with open(map_file, 'r') as f:
    coco_idx_map = {int(k)-1: int(v) for k, v in json.load(f).items()}

things_map = { 1: 11, 2: 18, 3: 13, 4: 17, 6: 15, 7: 16, 8: 14, 10: 6, 13: 7 }
stuff_map = { 100: 0, 123: 1, 91: 2, 129: 2, 109: 3, 110: 3, 111: 3, 112: 3, 131: 3, 117: 4, 116: 8, 125: 8, 126: 9, 119: 10 }

def bridge_to_cs(pred_tensor):
    res = torch.full_like(pred_tensor, 19)
    for idx, coco_id in coco_idx_map.items():
        if coco_id in things_map: res[pred_tensor == idx] = things_map[coco_id]
    for stuff_id, cs_id in stuff_map.items():
        res[pred_tensor == stuff_id] = cs_id
    return res

# 4. Initialize Cityscapes DataModule
from datasets.cityscapes_semantic import CityscapesSemantic
data_path = os.path.join(eomt_folder, 'data')
dm_cs = CityscapesSemantic(path=data_path, batch_size=1, num_workers=0)
dm_cs.setup()

# 5. Run Evaluation
evaluator = iouEval(20) # 19 classes + 1 ignore
for batch in tqdm(dm_cs.val_dataloader(), desc="Zero-Shot Baseline"):
    imgs, targets = batch
    gt = model_coco.to_per_pixel_targets_semantic(targets, 19)[0].to(device)

    with torch.no_grad():
        # Pre-process image (resize to 640 for COCO model)
        tx = model_coco.resize_and_pad_imgs_instance_panoptic([imgs[0].to(device)])
        mp, cp = model_coco(tx)
        mp = model_coco.revert_resize_and_pad_logits_instance_panoptic(
            F.interpolate(mp[-1], model_coco.img_size, mode="bilinear"),
            [imgs[0].shape[-2:]]
        )
        pred = model_coco.to_per_pixel_preds_panoptic(mp, cp[-1], model_coco.stuff_classes, 0.8, 0.8)[0][..., 0]

        # Map and Add to Evaluator
        evaluator.addBatch(bridge_to_cs(pred).unsqueeze(0).unsqueeze(0), gt.unsqueeze(0).unsqueeze(0))

_, ious = evaluator.getIoU()
print(f"\nZero-Shot Baseline mIoU: {ious[:19].mean()*100:.2f}%")

Zero-Shot Baseline: 100%|██████████| 500/500 [04:58<00:00,  1.67it/s]


Zero-Shot Baseline mIoU: 45.93%


## 3. Preparing for Fine-Tuning

### The Resolution Challenge
The COCO pre-trained weights were trained at **640x640**. If we want to fine-tune on Cityscapes (often at 1024x1024 or similar), we must handle the **positional embeddings** mismatch in the ViT encoder.

In [ ]:
from training.mask_classification_semantic import MaskClassificationSemantic

# 1. Setup Target Configuration (Cityscapes Semantic)
num_classes = 19
target_img_size = (640, 640) # We stay at 640 for initial loading stability

encoder_ft = ViT(img_size=640, backbone_name="vit_base_patch14_reg4_dinov2")
network_ft = EoMT(
    num_classes=num_classes,
    encoder=encoder_ft,
    num_q=200, # Changed from 100 to 200 to match the checkpoint's number of queries
    num_blocks=3,
    masked_attn_enabled=True
)

# 2. Initialize the Semantic Wrapper (The "Professor's" native class)
model_ft = MaskClassificationSemantic(
    network=network_ft,
    img_size=(640, 640),
    num_classes=num_classes,
    load_ckpt_class_head=False, # We don't want the 133-class head!
    ckpt_path=bin_path,          # Let the library handle the surgery!
    attn_mask_annealing_enabled=True # Added the missing argument
)

print("✅ Model surgically initialized for Fine-Tuning!")
print(f"Target classes: {model_ft.num_classes}")
print(f"Backbone loaded from: {bin_path}")

✅ Model surgically initialized for Fine-Tuning!
Target classes: 19
Backbone loaded from: /content/drive/MyDrive/FundGitHubProject/eomt/eomt_weights/eomt_coco.bin


### Debugging the Input Size vs Model Size

If you try to run with `img_size=1024` while loading 640x640 weights, the `pos_embed` will mismatch. Here is how the codebase handles it under the hood (or how you can manually verify it):

In [ ]:
# Peek into the positional embeddings
ckpt_raw = torch.load(bin_path, map_location="cpu")
state_dict = ckpt_raw.get('state_dict', ckpt_raw)
pos_embed_ckpt = state_dict['network.encoder.backbone.pos_embed']
print(f"Checkpoint Pos Embed shape: {pos_embed_ckpt.shape}")
# Shape is [1, 1600, 768] -> 40x40 grid (640/16 = 40)

# If we wanted 1024, we would need 64x64 = 4096 tokens.
expected_tokens_1024 = (1024 // 16) ** 2
print(f"Tokens needed for 1024x1024: {expected_tokens_1024}")

# SUGGESTION: Stay at 640 for the start of fine-tuning to preserve the pre-trained spatial knowledge.
# If you must go higher, the library's 'ViT' class should be modified to interpolate during loading.

Checkpoint Pos Embed shape: torch.Size([1, 1600, 768])
Tokens needed for 1024x1024: 4096


## 4. Fine Tuning
Now that the model is loaded with pre-trained backbone/transformer weights and a fresh Cityscapes head we can start fine tuning

In [ ]:
!pip install -U "torchao>=0.16.0" > /dev/null # peft requires an upgrade of torchao
!pip install peft > /dev/null

print("PEFT library installed successfully.")

PEFT library installed successfully.


### Applying LoRA
Now we configure the `LoraConfig`. For a Vision Transformer (ViT), we typically target the `qkv` (query, key, value) projection layers in the attention blocks.

In [ ]:


from peft import LoraConfig, get_peft_model

# Define LoRA Configuration
lora_config = LoraConfig(
    r=16,                  # Rank of the update matrices
    lora_alpha=32,         # Scaling factor
    target_modules=["qkv"], # Target the attention projection layers in the ViT
    lora_dropout=0.05,
    bias="none",
)


# Apply LoRA to the ViT encoder inside our network
model_ft.network.encoder = get_peft_model(model_ft.network.encoder, lora_config)

# Verify the trainable parameters
model_ft.network.encoder.print_trainable_parameters()

trainable params: 589,824 || all params: 87,487,488 || trainable%: 0.6742


### 4.1 Save LoRA Configuration

In [ ]:
import yaml
import os

# Create directory
lora_dir = os.path.join(eomt_folder, 'training', 'LoraConfig')
os.makedirs(lora_dir, exist_ok=True)
lora_yaml_path = os.path.join(lora_dir, 'lora_config.yaml')

# Extract relevant properties from the LoraConfig object
config_dict = {
    'r': lora_config.r,
    'lora_alpha': lora_config.lora_alpha,
    'target_modules': list(lora_config.target_modules) if isinstance(lora_config.target_modules, set) else lora_config.target_modules,
    'lora_dropout': lora_config.lora_dropout,
    'bias': lora_config.bias
}

# Save to YAML
with open(lora_yaml_path, 'w') as f:
    yaml.dump(config_dict, f, default_flow_style=False)

print(f"✅ LoRA config successfully saved to: {lora_yaml_path}")

✅ LoRA config successfully saved to: /content/drive/MyDrive/FundGitHubProject/eomt/training/LoraConfig/lora_config.yaml


### 4.3 Setup PyTorch Lightning Trainer
We set up the `Trainer` with `LearningRateMonitor` (so you can visualize the scheduler's effect) and mixed-precision for faster fine-tuning.

In [ ]:
!pip install wandb > /dev/null
import os
import wandb
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger
from google.colab import userdata

# 0. Close any active wandb runs to prevent conflicts
wandb.finish()

# 1. Authenticate with wandb using Colab secrets
# Ensure your secret is named 'WANDB_API_KEY' in the left panel 🔑
wandb_api_key = userdata.get('WANDB_API_KEY')
wandb.login(key=wandb_api_key)

# 2. Callbacks
lr_monitor = LearningRateMonitor(logging_interval='step')

checkpoint_callback = ModelCheckpoint(
    dirpath=os.path.join(project_root, 'checkpoints', 'cityscapes_lora'),
    filename='eomt-lora-{epoch:02d}-{val_iou:.2f}',
    save_top_k=2,
    monitor='metrics/val_iou_all',
    mode='max'
)

# 3. Set up the WandbLogger
wandb_logger = WandbLogger(project="eomt-cityscapes-finetuning", name="lora-run")

# 4. Extract max_epochs from the original coco config (defaulting to 10 if not found)
epochs = coco_config.get('trainer', {}).get('max_epochs', 10)

# 5. Initialize Trainer
trainer = Trainer(
    max_epochs=epochs,
    accelerator="auto",
    devices=1,
    callbacks=[lr_monitor, checkpoint_callback],
    logger=wandb_logger, # <-- Attach the logger here!
    precision="16-mixed",
    log_every_n_steps=10
)

print(f"✅ Trainer initialized successfully for {epochs} epochs with WandB!")

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


✅ Trainer initialized successfully for 24 epochs with WandB!


In [ ]:
import types
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# 1. Define a custom optimizer configuration for our model
def custom_configure_optimizers(self):
    # Strictly filter only parameters that require gradients (LoRA + new heads)
    trainable_params = [p for p in self.parameters() if p.requires_grad]

    # Setup AdamW optimizer (learning rate can be adjusted as needed)
    optimizer = AdamW(trainable_params, lr=2e-4, weight_decay=0.01)

    # Setup a learning rate scheduler (Cosine Annealing is a robust default)
    scheduler = CosineAnnealingLR(optimizer, T_max=self.trainer.max_epochs)

    return {
        "optimizer": optimizer,
        "lr_scheduler": {
            "scheduler": scheduler,
            "interval": "epoch", # Step the scheduler every epoch
            "frequency": 1
        }
    }

# 2. Inject this method into our PyTorch Lightning model instance
model_ft.configure_optimizers = types.MethodType(custom_configure_optimizers, model_ft)

# Print a summary of what's being trained
total_trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f"✅ Optimizer configured successfully!")
print(f"Total trainable parameters passed to optimizer: {total_trainable:,}")

# The next step will be: trainer.fit(model_ft, datamodule=dm_cs)

✅ Optimizer configured successfully!
Total trainable parameters passed to optimizer: 7,267,604


In [ ]:
print("🚀 Launching in-notebook Fine-Tuning routine with WandB...")
print("Using the LoRA-injected model (frozen backbone) and custom optimizer.")

# Re-initialize DataModule.
# CRITICAL: num_workers=0 prevents deadlocks when reading datasets from Google Drive!
# FIX: Reduced batch_size to 1 to prevent CUDA OutOfMemoryError
dm_cs = CityscapesSemantic(path=data_path, batch_size=4, num_workers=4, img_size=(640, 640))
dm_cs.setup()

# Skip the initial silent validation sanity check so we don't stall waiting for Drive I/O
# To change see
trainer.num_sanity_val_steps = 0

# FIX: Disable attention mask annealing since the step arrays were not provided in the config
model_ft.attn_mask_annealing_enabled = False

# FIX: Bypass the hardcoded WandB image logging inside the model to prevent SummaryWriter crash
import types
model_ft.plot_semantic = types.MethodType(lambda *args, **kwargs: None, model_ft)

# FIX: Explicitly set the model to train mode to activate LoRA dropouts and suppress the eval warning.
model_ft.train()

# Instead of the CLI, we use the `trainer` and `model_ft` you already
# carefully configured in the previous cells to preserve the PEFT "silencing"!

trainer.fit(
    model=model_ft,
    datamodule=dm_cs
)

🚀 Launching in-notebook Fine-Tuning routine with WandB...
Using the LoRA-injected model (frozen backbone) and custom optimizer.


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network   │ EoMT                   │ 94.2 M │ train │     0 │
│ 1 │ criterion │ MaskClassificationLoss │      0 │ train │     0 │
│ 2 │ metrics   │ ModuleList             │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 7.3 M                                                                                            
Non-trainable params: 86.9 M                                                                                       
Total params: 94.2 M                                                                                               
Total estimated model params size (MB): 376                                                                        
Modules in train mode: 426                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

INFO: mIoU: 69.2
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 69.2
INFO: mIoU: 70.1
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 70.1
INFO: mIoU: 72.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 72.5
INFO: mIoU: 74.0
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 74.0
INFO: mIoU: 74.9
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 74.9
INFO: mIoU: 75.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.5
INFO: mIoU: 75.1
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.1
INFO: mIoU: 75.6
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.6
INFO: mIoU: 75.6
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.6
INFO: mIoU: 75.8
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.8
INFO: mIoU: 75.4
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.4
INFO: mIoU: 76.3
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 76.3
INFO: mIoU: 76.6
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 76.6
INFO: mIoU: 75.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 75.5
INFO: 

## Still missing

- Learn how to load the checkpoints (/content/ProjectFolder/eomt/eomt_weights/finetuned)  
- Restore Layer learning rate decay in the encoder
- Discover the optimizer config we are supposed to use
- Try the polyschedure
- model_ft.attn_mask_annealing_enabled